# Classification with a linear layer

The simplest possible neural network: a single linear layer that learns to
classify 2D points into 3 classes — the idris-ml equivalent of a PyTorch
hello-world classifier.

**What you'll see:**
- Type-safe model construction with compile-time dimension checking
- The typed data surface (`Dataset`, `DataStream`)
- How the compiled example trains the same model with `fitSupervised`

**CLI equivalent:** `make example-supervised` (1000 epochs, full eval output)


## Architecture

A `Linear` layer maps input dimension `i` to output dimension `o` via
`y = Wx + b`. The type system enforces that the network's input and output
dimensions match the data.


In [1]:
:t linear

Nn.Linear.linear : KnownGrad g => Backend ex dt => Init (Linear i o ex dt g)


In [2]:
:t Seq

Nn.Seq.Seq : Nat -> Nat -> (0 _ : Executor) -> (0 _ : DType) -> (0 _ : GradMode) -> Type


`linear {i} {o}` is an `Init` builder: running it allocates the layer's weight
and bias and registers them with the C param registry. `Seq` chains layers into
a model, pinning only the endpoint dimensions. Build the single layer that maps
2D input to 3 classes:


In [3]:
:exec run (do { m <- runInitL (linear {i=2} {o=3} {ex=TapeExecutor} {dt=F64} {g=WithGrad}); discard m; liftIO1 (putStrLn "Model built (2 -> 3).") })

Model built (2 -> 3).


`runInitL` realises the `Init` action and derives parameter names from the
scope path (the PyTorch `state_dict` convention). Parameters reach the
optimizer only through the registry it populates: a tensor built outside
`Init` is invisible to the optimizer.


## Data

Five 2D points with one-hot encoded class labels. The decision function is
`argmax(x - y - 10, -4x + y + 5, 2x + y - 11)` which creates three regions
in the plane.


In [4]:
:doc Dataset

record Dataset.Dataset : Type -> Type
  Indexed data source. `item` takes a `Fin size`, so out-of-bounds
  access is unrepresentable — no runtime bounds check, no partiality.
  Totality: total
  Visibility: public export
  Constructor: MkDataset : (size : Nat) -> (Fin size -> IO sample) -> Dataset sample
  Projections:
    .item : (rec : Dataset sample) -> Fin (size rec) -> IO sample
    .size : Dataset sample -> Nat


In [5]:
:t fromVect

Dataset.fromVect : Vect n sample -> Dataset sample


A sample is a typed tensor pair: for a 2-feature, 3-class point the input is
`Tensor [2]` and the one-hot target is `Tensor [3]`. The compiled example
materialises fresh device tensors per access with `fromIndexed`, streams them
in order, and collates the five points into one `([5,2], [5,3])` batch C-side:

```idris
buildStream : IO (DataStream (Tensor [5, 2] Ex F NoGrad, Tensor [5, 3] Ex F NoGrad))
buildStream = do
  s <- stream NoShuffle (fromIndexed 5 sampleAt)
  pure (batched {b=5} {i=2} {o=3} s)
```

One batch of five points per epoch means one optimizer step per epoch:
full-batch gradient descent, matching the PyTorch reference's reduction.


## Training and evaluation

The compiled example threads the model through `fitSupervised` (tutorial
[04 Training](../tutorials/04_training.ipynb) covers the driver) and then
evaluates with `eval`, which retypes the model `WithGrad -> NoGrad`:

```idris
model <- runInitL (linear {i=2} {o=3})
bs <- liftIO1 buildStream
(MkBang (epochsDone, finalLoss) # trained) <-
  fitSupervised opt nllLossDefaultL bs (simpleConfig cfg.epochs) model
```

See `packages/idris-ml-examples/src/Example/Supervised.idr` for the loss
function and the eval loop.


## Type safety

The key guarantee: if the code compiles, the dimensions match. Try changing
`{i=2, o=3}` to `{i=2, o=4}` and the data points will fail to type-check
because the one-hot targets are `Vector 3 Double`, not `Vector 4 Double`.

This is checked at compile time with zero runtime cost.


## PyTorch comparison

The equivalent PyTorch code:

```python
model = nn.Linear(2, 3)
optimizer = optim.SGD(model.parameters(), lr=0.1)

for epoch in range(1000):
    output = model(input_tensor)
    loss = F.cross_entropy(output, target_tensor)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
```

The main differences:
- PyTorch dimensions are checked at runtime; idris-ml checks them at compile time.
- PyTorch requires manual `zero_grad()` / `backward()` / `step()` sequencing;
  idris-ml's `trainStep` runs zero_grad → backward → clip → step in one call.
- PyTorch's `model.parameters()` is collected dynamically; idris-ml's `runInitL`
  registers parameters at construction.

See `pytorch/torch_ref/scripts/supervised.py` for the full reference implementation.


Next: [RNN and LSTM](rnn_lstm.ipynb) — recurrent models for sequential patterns.
